# 時代別のコーパスを作って頻度が変わった単語を調べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_lexical_semantics/diachronic_corpus.ipynb)

NLP クックブック 20-08 ／ 語の意味と意味の変化

書かれた時代の違うテキストを集めて、時代によって使われる頻度が大きく変わった単語を調べたい。

In [1]:
!pip install -q datasets fugashi unidic-lite

In [2]:
import math, random, re
from collections import Counter
from datasets import load_dataset
from fugashi import Tagger

ds = load_dataset("globis-university/aozorabunko-clean", split="train")     # 青空文庫の約 1.7 万作品 (約 240MB)
print(ds[0]["meta"]["作品名"], ds[0]["meta"]["姓"], ds[0]["meta"]["生年月日"], ds[0]["meta"]["文字遣い種別"])

def era(meta):
    """作者の生年で時代を分ける。表記の違いが混ざらないよう、新字新仮名の作品だけを使う"""
    year = (meta.get("生年月日") or "")[:4]
    if meta["文字遣い種別"] != "新字新仮名" or not year.isdigit():
        return None
    return "旧" if 1850 <= int(year) < 1880 else "新" if 1895 <= int(year) < 1930 else None

texts = {"旧": [], "新": []}
for ex in ds:
    if era(ex["meta"]):
        texts[era(ex["meta"])].append(ex["text"])
print({e: len(t) for e, t in texts.items()})

ウェストミンスター寺院 アーヴィング 1783-04-03 新字新仮名
{'旧': 2035, '新': 4063}


In [3]:
tagger = Tagger()
random.seed(0)

def count_words(works, max_chars=10_000_000):
    counts, n = Counter(), 0
    for text in random.sample(works, len(works)):
        text = text[:30000]                                   # 長編 1 作品に偏らないよう、各作品の冒頭だけを使う
        in_work = Counter(w.surface for w in tagger(text) if w.feature.pos1 == "名詞" and w.feature.pos2 == "普通名詞")
        counts.update({w: min(c, 5) for w, c in in_work.items()})     # 1 作品の中で何度出ても 5 回までしか数えない (登場人物名などの偏りを抑える)
        n += len(text)
        if n > max_chars:
            break
    return counts

freq = {e: count_words(t) for e, t in texts.items()}
total = {e: sum(c.values()) for e, c in freq.items()}
print(total)

{'旧': 1033505, '新': 948328}


In [4]:
def log_ratio(word, smooth=5):
    """100 万語あたりの頻度の比の対数 (正なら新しい時代に増えた)。頻度 0 で発散しないよう、少し足してから比べる"""
    old = (freq["旧"][word] + smooth) / total["旧"]
    new = (freq["新"][word] + smooth) / total["新"]
    return math.log2(new / old)

vocab = [w for w in set(freq["旧"]) | set(freq["新"]) if freq["旧"][w] + freq["新"][w] >= 50 and len(w) >= 2]
ranked = sorted(vocab, key=log_ratio)
fmt = lambda ws: " ".join(f"{w}({freq['旧'][w]}→{freq['新'][w]})" for w in ws)
print("増えた語:", fmt(ranked[::-1][:30]))
print("減った語:", fmt(ranked[:30]))
for w in ["ラジオ", "映画", "放送", "電話", "汽車", "馬車", "書生", "細君"]:
    print(f"{w}: 100 万語あたり {freq['旧'][w] / total['旧'] * 1e6:.0f} → {freq['新'][w] / total['新'] * 1e6:.0f} (対数比 {log_ratio(w):+.2f})")

増えた語: リアリズム(1→96) バカ(7→188) 疎開(1→91) プロレタリア(8→169) プロレタリアート(1→70) いっぺん(1→67) 資源(0→55) 全て(1→62) フシギ(1→61) 感性(0→50) 職場(3→82) 捜査(7→125) ファシズム(1→59) ムリ(1→55) 爆撃(2→65) ぜんたい(3→75) 共産(7→115) 動員(4→80) とし(1→50) 左翼(2→57) 課題(9→119) 失業(3→65) 爆弾(6→91) 空襲(7→96) 勤労(12→134) スイッチ(2→52) ウチ(3→60) ワケ(3→60) アパート(7→89) 実践(8→94)
減った語: 親仁(66→0) 浅葱(56→0) 年紀(54→0) 漢学(81→3) 起原(55→1) 事がら(58→2) 俳人(65→3) 参詣(181→17) 差支(68→4) 楽み(65→4) 月影(55→3) 称呼(55→3) 平生(159→17) 在来(69→5) 小児(202→24) 本邦(52→3) 寂寞(65→5) 黒髪(106→11) 料簡(92→9) 花崗(56→4) 上記(49→3) 多年(123→14) 猪口(62→5) 深切(48→3) 談話(145→19) 俳句(169→23) 出品(87→10) 何某(50→4) 時節(135→18) 因襲(49→4)
ラジオ: 100 万語あたり 25 → 188 (対数比 +2.69)
映画: 100 万語あたり 125 → 391 (対数比 +1.61)
放送: 100 万語あたり 24 → 147 (対数比 +2.39)
電話: 100 万語あたり 109 → 348 (対数比 +1.63)
汽車: 100 万語あたり 397 → 370 (対数比 -0.10)
馬車: 100 万語あたり 249 → 133 (対数比 -0.88)
書生: 100 万語あたり 185 → 76 (対数比 -1.22)
細君: 100 万語あたり 213 → 198 (対数比 -0.10)


- 意味の変化を調べる前に、まず頻度の変化を見る。新しい事物の登場（ラジオ、映画）や、すたれた事物（馬車、書生）の減少は頻度に表れる。ただし、汽車や細君のように、ほとんど変わらない語もある。
- 時代の情報として、ここでは作者の生年を使った。作品の発表年が分かるコーパス（新聞、雑誌、国会会議録）なら、より正確に分けられる。
- 時代による違いと、ジャンルや作者の違いは混ざる。特定の作品にしか出ない語（登場人物の名前など）が上位に来やすいので、各作品から使う量をそろえ、1 作品あたりの回数に上限を設け、固有名詞を除いている。
- 形態素解析の単位にも注意する。UniDic では「自動車」が「自動 / 車」に分かれるので、そのままでは数えられない。
- 日本語の本格的な通時コーパスには、国立国語研究所の日本語歴史コーパス (CHJ) や昭和・平成書き言葉コーパスがある（利用には申請が必要）。